In [1]:
import os
import sys
from pathlib import Path

# This notebook lives in `kazemo/`. Make imports work even when run from inside that folder.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "kazemo":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["DATASETS_AUDIO_BACKEND"] = "soundfile"
os.environ["TORCHCODEC_QUIET"] = "1"

from kazemo.load_data import load_kazemotts

# Keep this small for quick iteration; set to None for full dataset
MAX_SAMPLES = 2000

dataset = load_kazemotts(max_samples=MAX_SAMPLES)

# Prefer an eval-ish split name when present; fall back to first.
TEST_SPLIT = next(
    (s for s in ("test", "validation", "val") if s in dataset),
    next(iter(dataset.keys())),
)

print("Splits:", list(dataset.keys()))
for k in dataset.keys():
    print(f"  {k}: {len(dataset[k])} rows, columns: {dataset[k].column_names}")
print(f"\nUsing '{TEST_SPLIT}' as the evaluation split.")


Loading KazEmoTTS from zip (skip load_dataset)...


/home/fatikh/ML/ML/lib/python3.12/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Repo ready. Locating zip...
Found EmoKaz.zip (10.5 GB).
Using already-extracted files.
Scanning for audio files...
Using 2000 audio files.
Preparing split 'train' (2000 rows)...


Adding emotion labels:   0%|          | 0/2000 [00:00<?, ? examples/s]

Splits: ['train']
  train: 2000 rows, columns: ['audio', 'emotion']

Using 'train' as the evaluation split.


In [2]:
from inference import resolve_checkpoint, load_model, run_row_inference
from load_data import read_audio
from train import SAMPLE_RATE
import numpy as np
from IPython.display import display, Audio

ckpt_path = resolve_checkpoint()
print(f"Using checkpoint: {ckpt_path}")

model, emotion_encoder, gender_encoder, age_encoder, \
    id2emotion, id2gender, id2age, processor = load_model(ckpt_path)
print("Model ready.")


Using checkpoint: /home/fatikh/issai/audio-clf/models/finetune/best_model_finetuned.pt
Model ready.


In [3]:
def check_row(index: int, split: str | None = None):
    split = split or TEST_SPLIT
    row = dataset[split][index]

    audio_data, sample_rate = read_audio(row["audio"])
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)
    duration_sec = len(audio_data) / sample_rate
    audio_widget = Audio(data=audio_data, rate=sample_rate, normalize=True, embed=True)

    # ── Inference (same path as demo.py via inference.run_row_inference) ───────
    results = run_row_inference(row, model, processor, id2emotion, id2gender, id2age)

    # ── Print summary ─────────────────────────────────────────────────────────
    sep = "─" * 56
    print(sep)
    print(f"  Row {index}  |  split: '{split}'  |  {duration_sec:.2f}s")
    print(f"  Emotion gt  : {row.get('emotion', '—')}")
    print(sep)

    r = results["emotion"]
    match = "✓" if r["pred"] == str(row.get("emotion")) else "✗"
    print(
        f"\n  Emotion    pred: {r['pred']} ({r['confidence']*100:.1f}%)  {match}   gt: {row.get('emotion', '—')}"
    )
    print(f"            top-3: {[(lbl, f'{p*100:.1f}%') for lbl, p in r['top3']]}")

    # No ground-truth gender/age in KazEmoTTS, but we can still inspect predictions
    for task in ("gender", "age"):
        rt = results[task]
        print(f"\n  {task.capitalize():<10} pred: {rt['pred']} ({rt['confidence']*100:.1f}%)")
        print(f"             top-3: {[(lbl, f'{p*100:.1f}%') for lbl, p in rt['top3']]}")

    print(f"\n{sep}")
    display(audio_widget)
    return audio_widget, results


In [7]:
# Change index to inspect any row from the evaluation split
audio, results = check_row(index=3)


────────────────────────────────────────────────────────
  Row 3  |  split: 'train'  |  6.69s
  Emotion gt  : angry
────────────────────────────────────────────────────────

  Emotion    pred: neutral (81.6%)  ✗   gt: angry
            top-3: [('neutral', '81.6%'), ('happy', '10.5%'), ('sad', '4.8%')]

  Gender     pred: M (73.9%)
             top-3: [('M', '73.9%'), ('F', '26.1%')]

  Age        pred: adult (81.5%)
             top-3: [('adult', '81.5%'), ('senior', '10.0%'), ('young', '5.6%')]

────────────────────────────────────────────────────────


In [5]:
# List some indices by ground-truth emotion (uses the parsed emotion label)
split = dataset[TEST_SPLIT]

from collections import defaultdict

by_emo = defaultdict(list)
for i in range(len(split)):
    by_emo[split[i]["emotion"]].append(i)

for emo in sorted(by_emo.keys()):
    print(f"{emo:<10}: {len(by_emo[emo])} rows | examples: {by_emo[emo][:10]}")


angry     : 349 rows | examples: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
fearful   : 282 rows | examples: [330, 331, 332, 333, 334, 335, 336, 337, 338, 339]
happy     : 358 rows | examples: [612, 613, 614, 615, 616, 617, 618, 619, 620, 621]
neutral   : 373 rows | examples: [970, 971, 972, 973, 974, 975, 976, 977, 978, 979]
sad       : 329 rows | examples: [1343, 1344, 1345, 1346, 1347, 1348, 1349, 1350, 1351, 1352]
surprised : 309 rows | examples: [1672, 1673, 1674, 1675, 1676, 1677, 1678, 1679, 1680, 1681]
